In [ ]:
# ======================================================
# STEP 2: Import dependencies
# ======================================================
import pandas as pd
import numpy as np
import re
import nltk
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import fasttext
from gensim.models import FastText, Word2Vec
from torch import nn
import torch
from torch.utils.data import DataLoader, Dataset


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

# Path to your folder in Google Drive
folder_path = "/content/drive/MyDrive/Mini"

# Load all four CSVs
fake_politifact = pd.read_csv(os.path.join(folder_path, "politifact_fake.csv"))
real_politifact = pd.read_csv(os.path.join(folder_path, "politifact_real.csv"))
fake_gossipcop = pd.read_csv(os.path.join(folder_path, "gossipcop_fake.csv"))
real_gossipcop = pd.read_csv(os.path.join(folder_path, "gossipcop_real.csv"))
welfake = pd.read_csv(os.path.join(folder_path, "WELFake_Dataset.csv"))
news = pd.read_csv(os.path.join(folder_path, "news.csv"))


# Add labels: 0 = fake, 1 = real
fake_politifact["label"] = 0
real_politifact["label"] = 1
fake_gossipcop["label"] = 0
real_gossipcop["label"] = 1

# Merge into one DataFrame
df1 = pd.concat([fake_politifact, real_politifact, fake_gossipcop, real_gossipcop])
df2 = pd.concat([welfake, news])
df1["combined_text"] = df1["title"].astype(str)
df1 = df1[["combined_text", "label"]]

df2["combined_text"] = df2["title"].astype(str) + " " + df2["text"].astype(str)
df2 = df2[["combined_text", "label"]]

df = pd.concat([df1, df2])
# Combine the two DataFrames

# Keep only useful columns

# Save combined dataset
output_path = os.path.join(folder_path, "FakeNewsNet_clean.csv")
df.to_csv(output_path, index=False)

print("✅ Combined dataset saved at:", output_path)
print("📊 Total samples:", len(df))
print(df.columns)
print(fake_politifact.columns)
print(fake_gossipcop.columns)
print(real_politifact.columns)
print(real_gossipcop.columns)
print(welfake.columns)
print(news.columns)


In [ ]:
# ======================================================
# STEP 4: Preprocessing
# ======================================================
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = re.sub(r'http\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text) # remove special chars
    text = text.lower()
    text = " ".join([w for w in text.split() if w not in stop_words])
    return text

df['clean_text'] = df['text'].apply(preprocess)

X = df['clean_text']
y = df['label']


In [ ]:
# ======================================================
# STEP 5: Train/Test Split
# ======================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# ======================================================
# STEP 6: FastText Embeddings (Unsupervised)
# ======================================================
with open("train_text.txt", "w") as f:
    for line in X_train:
        f.write(line + "\n")

ft_model = fasttext.train_unsupervised("train_text.txt", model="skipgram")


In [ ]:
# ======================================================
# STEP 7: ML Models with FastText embeddings
# ======================================================
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def get_vector(text):
    return np.mean([ft_model[w] for w in text.split() if w in ft_model], axis=0)

X_train_vec = np.array([get_vector(t) for t in X_train])
X_test_vec = np.array([get_vector(t) for t in X_test])

svm = SVC()
svm.fit(X_train_vec, y_train)
y_pred_svm = svm.predict(X_test_vec)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))


In [ ]:
# ======================================================
# STEP 8: Deep Learning - CNN-LSTM with FastText embeddings
# ======================================================
class NewsDataset(Dataset):
    def __init__(self, texts, labels, ft_model, max_len=100):
        self.texts = texts
        self.labels = labels.values
        self.ft_model = ft_model
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = self.texts.iloc[idx].split()[:self.max_len]
        vecs = [self.ft_model[w] for w in words if w in self.ft_model]
        if len(vecs) < self.max_len:
            vecs += [np.zeros(100)] * (self.max_len - len(vecs))
        return torch.tensor(vecs, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

train_ds = NewsDataset(X_train, y_train, ft_model)
test_ds = NewsDataset(X_test, y_test, ft_model)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)


In [ ]:
# ======================================================
# STEP 9: Define CNN-LSTM Model
# ======================================================
class CNN_LSTM(nn.Module):
    def __init__(self):
        super(CNN_LSTM, self).__init__()
        self.conv1 = nn.Conv1d(100, 128, kernel_size=5)
        self.pool = nn.MaxPool1d(2)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.fc = nn.Linear(128, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.permute(0, 2, 1)  
        x = self.pool(self.relu(self.conv1(x)))
        x = x.permute(0, 2, 1)
        _, (h, _) = self.lstm(x)
        out = self.fc(h[-1])
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN_LSTM().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# ======================================================
# STEP 10: Train CNN-LSTM
# ======================================================
for epoch in range(3):  
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
# ======================================================
# STEP 11: Evaluate CNN-LSTM
# ======================================================
model.eval()
y_pred, y_true = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, axis=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(y_batch.numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))


In [ ]:
# ======================================================
# STEP 12: Interpretability - LDA
# ======================================================
from gensim import corpora
from gensim.models.ldamodel import LdaModel

texts = [t.split() for t in X_train]
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

lda = LdaModel(corpus=corpus, num_topics=5, id2word=dictionary, passes=10)

for idx, topic in lda.print_topics(-1):
    print(f"Topic {idx}: {topic}")


In [ ]:
# ======================================================
# STEP 13: Interpretability - LIME
# ======================================================
from lime.lime_text import LimeTextExplainer
from sklearn.pipeline import make_pipeline

# Wrap SVM as example
pipeline = make_pipeline(
    lambda texts: np.array([get_vector(t) for t in texts]), svm
)

explainer = LimeTextExplainer(class_names=["Fake","Real"])
idx = 10
exp = explainer.explain_instance(X_test.iloc[idx], pipeline.predict_proba, num_features=10)
exp.show_in_notebook(text=True)
